# Active Learning Experiment

Сравнение стратегий `entropy` vs `random` на одном датасете.

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd
from agents.al_agent import ActiveLearningAgent
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/labeled/labeled_dataset.csv')
if 'auto_label' not in df.columns:
    df['auto_label'] = df.get('label', 'unlabeled')
df[['text','auto_label']].head()

In [ ]:
label_col = 'auto_label'
strat = df[label_col] if df[label_col].value_counts().min() >= 2 else None
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=strat)
n_start = min(50, max(10, len(train_df)//2))
strat_train = train_df[label_col] if train_df[label_col].value_counts().min() >= 2 else None
labeled_start, pool_df = train_test_split(train_df, train_size=n_start, random_state=42, stratify=strat_train)

agent = ActiveLearningAgent(model='logreg')
hist_entropy = agent.run_cycle(labeled_start, pool_df, strategy='entropy', n_iterations=5, batch_size=20, test_df=test_df)
hist_random = agent.run_cycle(labeled_start, pool_df, strategy='random', n_iterations=5, batch_size=20, test_df=test_df)
hist_entropy[-1], hist_random[-1]

In [ ]:
df_hist = pd.concat([pd.DataFrame(hist_entropy), pd.DataFrame(hist_random)], ignore_index=True)
plt.figure(figsize=(9,5))
for strat_name in sorted(df_hist['strategy'].unique()):
    s = df_hist[df_hist['strategy']==strat_name].sort_values('n_labeled')
    plt.plot(s['n_labeled'], s['f1'], marker='o', label=strat_name)
plt.xlabel('n_labeled')
plt.ylabel('F1')
plt.title('Entropy vs Random')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
savings = agent.compare({'entropy': hist_entropy, 'random': hist_random})
print(json.dumps(savings, ensure_ascii=False, indent=2))